In [1]:
import torch
import math
from transformers import (
    AutoModelForMaskedLM, 
    AutoTokenizer, 
    DataCollatorForLanguageModeling, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)
from transformers import RobertaConfig, RobertaTokenizerFast,RobertaForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from datasets import load_dataset

/home/ml/.conda/envs/ykmlen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
import pandas as pd
import numpy as np
from rdkit.Chem.MolStandardize import rdMolStandardize
from pandarallel import pandarallel

In [ ]:
data = pd.read_table('../../data/pfas_6892694.txt',sep='\t',names=['smiles'])

In [ ]:

pandarallel.initialize(progress_bar=False)

SALT_REMOVER = SaltRemover()
UNCHARGER = rdMolStandardize.Uncharger()
CHOOSER = rdMolStandardize.LargestFragmentChooser()

def standardize_smiles(smiles: str):
    if not isinstance(smiles, str) or not smiles.strip():
        return np.nan
    
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.nan
        mol = SALT_REMOVER.StripMol(mol, dontRemoveEverything=True)
        if "." in smiles:
            mol = CHOOSER.choose(mol)    
        if "+" in smiles or "-" in smiles:      
            mol = UNCHARGER.uncharge(mol) 
        return Chem.MolToSmiles(mol, isomericSmiles=True,canonical=True)
        
    except Exception:
        return np.nan

def process_smiles_data(df, smiles_col='smiles'):

    df['standardized_smiles'] = df[smiles_col].parallel_apply(standardize_smiles) 
    df.loc[:,'is_changed'] = (df[smiles_col] != df['standardized_smiles']) & df['standardized_smiles'].notna() 
    total = len(df)
    failed = df['standardized_smiles'].isna().sum()
    changed = df['is_changed'].sum()
    print("-" * 30)
    return df

INFO: Pandarallel will run on 64 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [ ]:
data = process_smiles_data(data,smiles_col='smiles')

In [ ]:
data.to_csv("pfas_smiles_processed.csv",index=None)

In [8]:
data=pd.read_csv("pfas_smiles_processed.csv")

In [5]:
pd.DataFrame(data["standardized_smiles"].unique()).to_csv("pfas_smiles_unique.txt",index=None,header=None)